# Paper-style OmniXAS training and all-elements eta table

Train/check the paper-style model families for:

- **FEFF**: Ti, V, Cr, Mn, Fe, Co, Ni, Cu
- **VASP**: Ti and Cu, when data/checkpoints are available

Every dataset/model has its own training flag below. The final table selects checkpoints by the exact validation loss stored inside each checkpoint, then reports test eta.

## 1. Imports, paths, hyperparameters, and training flags

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import random
import re

import numpy as np
import pandas as pd
import torch
from lightning.pytorch import seed_everything
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, TensorDataset

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.metrics import ModelMetrics
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

SEED = 42  # set to None for a fresh random seed
if SEED is None:
    SEED = random.SystemRandom().randint(0, 2**32 - 1)
os.environ["PYTHONHASHSEED"] = str(SEED)
seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision("high")

REPO_ROOT = Path.cwd()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").exists()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not find OmniXAS repo root.")
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
OUTPUT_ROOT = REPO_ROOT / "output" / "training"
HYDRA_TRAIN_CFG = OmegaConf.load(REPO_ROOT / "config" / "paper_hydra" / "train.yaml")
RESULTS_DIR = OUTPUT_ROOT / "comparisons" / "paper_reproduction_all" / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_seed{SEED}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM, OUTPUT_DIM = 64, 141
FEFF_ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
VASP_ELEMENTS = ["Ti", "Cu"]
UNIVERSAL_DIMS = [500, 500, 550]
FEFF_HPARAMS = {
    "Ti": {"batch_size": 32, "widths": [600, 600, 450]},
    "V":  {"batch_size": 32, "widths": [600, 550, 450]},
    "Cr": {"batch_size": 32, "widths": [450, 350, 150]},
    "Mn": {"batch_size": 64, "widths": [500, 400, 300]},
    "Fe": {"batch_size": 64, "widths": [450, 400, 450]},
    "Co": {"batch_size": 32, "widths": [600, 550, 450]},
    "Ni": {"batch_size": 32, "widths": [600, 300]},
    "Cu": {"batch_size": 32, "widths": [600, 600, 400]},
}
VASP_HPARAMS = {
    "Ti": {"batch_size": 64, "widths": [500, 600, 400]},
    "Cu": {"batch_size": 64, "widths": [550, 600, 450]},
}

MAX_EPOCHS = 1000
PATIENCE = 25
INITIAL_LR = 1e-3
# Tuned models use Hydra's fixed LR and skip the LR finder.
# Override this value manually here if you want to sweep, e.g. 3e-5 or 1e-5.
TUNED_INITIAL_LR = float(HYDRA_TRAIN_CFG.training.lr)
TUNED_SHUFFLE = True
MIN_LR = 1e-4
DEFAULT_DROPOUT = 0.5
TUNED_DROPOUT_SWEEP = [0.5, 0.0]
N_TRAINING_RUNS = 1

TRAIN_UNIVERSAL_FEFF = False

# ExpertXAS training flags.
TRAIN_TI_FEFF_EXPERT = False
TRAIN_V_FEFF_EXPERT = False
TRAIN_CR_FEFF_EXPERT = False
TRAIN_MN_FEFF_EXPERT = False
TRAIN_FE_FEFF_EXPERT = False
TRAIN_CO_FEFF_EXPERT = False
TRAIN_NI_FEFF_EXPERT = False
TRAIN_CU_FEFF_EXPERT = False
TRAIN_TI_VASP_EXPERT = False
TRAIN_CU_VASP_EXPERT = False

# Tuned-UniversalXAS fine-tuning flags.
TRAIN_TI_FEFF_TUNED = False
TRAIN_V_FEFF_TUNED = False
TRAIN_CR_FEFF_TUNED = False
TRAIN_MN_FEFF_TUNED = False
TRAIN_FE_FEFF_TUNED = False
TRAIN_CO_FEFF_TUNED = False
TRAIN_NI_FEFF_TUNED = False
TRAIN_CU_FEFF_TUNED = False
TRAIN_TI_VASP_TUNED = False
TRAIN_CU_VASP_TUNED = False

FEFF_EXPERT_FLAGS = {
    "Ti": TRAIN_TI_FEFF_EXPERT, "V": TRAIN_V_FEFF_EXPERT, "Cr": TRAIN_CR_FEFF_EXPERT, "Mn": TRAIN_MN_FEFF_EXPERT,
    "Fe": TRAIN_FE_FEFF_EXPERT, "Co": TRAIN_CO_FEFF_EXPERT, "Ni": TRAIN_NI_FEFF_EXPERT, "Cu": TRAIN_CU_FEFF_EXPERT,
}
FEFF_TUNED_FLAGS = {
    "Ti": TRAIN_TI_FEFF_TUNED, "V": TRAIN_V_FEFF_TUNED, "Cr": TRAIN_CR_FEFF_TUNED, "Mn": TRAIN_MN_FEFF_TUNED,
    "Fe": TRAIN_FE_FEFF_TUNED, "Co": TRAIN_CO_FEFF_TUNED, "Ni": TRAIN_NI_FEFF_TUNED, "Cu": TRAIN_CU_FEFF_TUNED,
}
VASP_EXPERT_FLAGS = {"Ti": TRAIN_TI_VASP_EXPERT, "Cu": TRAIN_CU_VASP_EXPERT}
VASP_TUNED_FLAGS = {"Ti": TRAIN_TI_VASP_TUNED, "Cu": TRAIN_CU_VASP_TUNED}

print("Seed:", SEED)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)
print("Tuned fine-tune LR from config/paper_hydra/train.yaml:", TUNED_INITIAL_LR)
print("Tuned fine-tune DataLoader shuffle:", TUNED_SHUFFLE)

## 2. Utilities

These small functions keep the training/evaluation cells readable:

- `load_split(element, spectrum_type)`: reads `train`, `val`, and `test` `X/y` text files for one dataset, then returns an `MLSplits` object.
- `run_root(kind, element=None, spectrum_type=None)`: gives the output folder where checkpoints for a model family live. For example, expert Cu FEFF goes under `output/training/expertXAS/Cu_FEFF/runs`.
- `new_run_dir(root, seed, dropout=None)`: creates a fresh timestamped run directory such as `paper_..._seed42` or `paper_..._seed42_dropout0p0`.
- `make_model(directory, hidden_dims, batch_size, ...)`: builds an `XASBlockRegressor` with standard paper training settings and optional fixed-LR/shuffle controls for tuned runs.
- `predict_direct(model, X)`: predicts with direct PyTorch inference instead of `model.predict()`. This avoids Lightning splitting predictions across multiple GPUs during evaluation.
- `checkpoint_val_loss(ckpt)`: reads the exact unrounded best validation loss stored inside a Lightning checkpoint. It falls back to the rounded filename only if needed.
- `best_universal_source_for(label)`: selects the UniversalXAS checkpoint with the lowest exact validation loss before fine-tuning.

In [ ]:
def load_split(element, spectrum_type):
    return MLSplits(**{
        name: MLData(
            X=np.loadtxt(DATA_DIR / f"{element}_{spectrum_type}_{name}_X.txt", dtype=np.float32),
            y=np.loadtxt(DATA_DIR / f"{element}_{spectrum_type}_{name}_y.txt", dtype=np.float32),
        )
        for name in ["train", "val", "test"]
    })


def run_root(kind, element=None, spectrum_type=None):
    if kind == "universal":
        return OUTPUT_ROOT / "universalXAS" / "All_FEFF" / "runs"
    folder = "expertXAS" if kind == "expert" else "tunedUniversalXAS"
    return OUTPUT_ROOT / folder / f"{element}_{spectrum_type}" / "runs"


def new_run_dir(root, seed, dropout=None):
    name = f"paper_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_seed{seed}"
    if dropout is not None:
        name += f"_dropout{str(dropout).replace('.', 'p')}"
    path = Path(root) / name
    path.mkdir(parents=True, exist_ok=False)
    return path


def make_model(
    directory,
    hidden_dims,
    batch_size,
    *,
    initial_lr=INITIAL_LR,
    use_lr_finder=True,
    shuffle=False,
):
    return XASBlockRegressor(
        directory=str(directory), overwrite_save_dir=False,
        input_dim=INPUT_DIM, output_dim=OUTPUT_DIM,
        hidden_dims=list(hidden_dims), batch_size=batch_size,
        max_epochs=MAX_EPOCHS, early_stopping_patience=PATIENCE,
        initial_lr=initial_lr, min_lr=MIN_LR,
        use_lr_finder=use_lr_finder,
        shuffle=shuffle,
    )


def predict_direct(model, X, batch_size=1024):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    module = model.model.to(device).eval()
    loader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (xb,) in loader:
            preds.append(module(xb.to(device)).detach().cpu().numpy())
    return np.concatenate(preds, axis=0)


def checkpoint_val_loss(ckpt_path):
    try:
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        except TypeError:
            ckpt = torch.load(ckpt_path, map_location="cpu")
        scores = []
        for callback_name, state in ckpt.get("callbacks", {}).items():
            if "ModelCheckpoint" not in str(callback_name):
                continue
            score = state.get("best_model_score")
            if score is not None:
                scores.append(float(score.detach().cpu().item() if torch.is_tensor(score) else score))
        if scores:
            return min(scores)
    except Exception as exc:
        print(f"Warning: could not read exact val_loss from {ckpt_path}: {exc}")

    match = re.search(r"val_loss[=_](\d+(?:\.\d+)?)", Path(ckpt_path).name)
    return float(match.group(1)) if match else float("inf")


def best_universal_source_for(label):
    ckpts = sorted(run_root("universal").glob("paper_*/best*.ckpt"))
    assert ckpts, f"No UniversalXAS checkpoints found in {run_root('universal')}"
    best_ckpt = min(ckpts, key=checkpoint_val_loss)
    print(f"UniversalXAS source for {label}: val_loss={checkpoint_val_loss(best_ckpt):.8g} -> {best_ckpt}")
    return best_ckpt.parent

## 3. Load all available paper data

In [ ]:
feff_splits = {element: load_split(element, "FEFF") for element in FEFF_ELEMENTS}
vasp_splits = {
    element: load_split(element, "VASP")
    for element in VASP_ELEMENTS
    if (DATA_DIR / f"{element}_VASP_test_X.txt").exists()
}
universal_parts = [feff_splits[element] for element in FEFF_ELEMENTS]
universal_feff = MLSplits(
    train=MLData(X=np.concatenate([s.train.X for s in universal_parts]), y=np.concatenate([s.train.y for s in universal_parts])),
    val=MLData(X=np.concatenate([s.val.X for s in universal_parts]), y=np.concatenate([s.val.y for s in universal_parts])),
    test=MLData(X=np.concatenate([s.test.X for s in universal_parts]), y=np.concatenate([s.test.y for s in universal_parts])),
)

print("Universal FEFF train/val/test:", universal_feff.train.X.shape, universal_feff.val.X.shape, universal_feff.test.X.shape)
for element in FEFF_ELEMENTS:
    split = feff_splits[element]
    print(f"{element} FEFF train/val/test:", split.train.X.shape, split.val.X.shape, split.test.X.shape)
for element, split in vasp_splits.items():
    print(f"{element} VASP train/val/test:", split.train.X.shape, split.val.X.shape, split.test.X.shape)

## 4. Train UniversalXAS FEFF

In [ ]:
if TRAIN_UNIVERSAL_FEFF:
    XASBlock.DROPOUT = DEFAULT_DROPOUT
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        seed_everything(train_seed, workers=True)
        run_dir = new_run_dir(run_root("universal"), train_seed)
        print(f"Training UniversalXAS FEFF run {run_number}/{N_TRAINING_RUNS}:", run_dir)
        make_model(run_dir, UNIVERSAL_DIMS, 32).fit(universal_feff)
else:
    print("Skipping UniversalXAS FEFF training")

## 5. Train FEFF ExpertXAS models

In [ ]:
for element, do_train in FEFF_EXPERT_FLAGS.items():
    if not do_train:
        continue
    hparams = FEFF_HPARAMS[element]
    XASBlock.DROPOUT = DEFAULT_DROPOUT
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        seed_everything(train_seed, workers=True)
        run_dir = new_run_dir(run_root("expert", element, "FEFF"), train_seed)
        print(f"Training {element} FEFF ExpertXAS run {run_number}/{N_TRAINING_RUNS}:", run_dir)
        make_model(run_dir, hparams["widths"], hparams["batch_size"]).fit(feff_splits[element])

## 6. Train VASP ExpertXAS models

In [ ]:
for element, do_train in VASP_EXPERT_FLAGS.items():
    if not do_train or element not in vasp_splits:
        continue
    hparams = VASP_HPARAMS[element]
    XASBlock.DROPOUT = DEFAULT_DROPOUT
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        seed_everything(train_seed, workers=True)
        run_dir = new_run_dir(run_root("expert", element, "VASP"), train_seed)
        print(f"Training {element} VASP ExpertXAS run {run_number}/{N_TRAINING_RUNS}:", run_dir)
        make_model(run_dir, hparams["widths"], hparams["batch_size"]).fit(vasp_splits[element])

## 7. Fine-tune UniversalXAS on FEFF targets

In [ ]:
for element, do_train in FEFF_TUNED_FLAGS.items():
    if not do_train:
        continue
    hparams = FEFF_HPARAMS[element]
    source_dir = best_universal_source_for(f"{element} FEFF tuning")
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        for dropout in TUNED_DROPOUT_SWEEP:
            seed_everything(train_seed, workers=True)
            XASBlock.DROPOUT = dropout
            run_dir = new_run_dir(run_root("tuned", element, "FEFF"), train_seed, dropout)
            print(
                f"Fine-tuning {element} FEFF Tuned-UniversalXAS "
                f"dropout={dropout} lr={TUNED_INITIAL_LR} shuffle={TUNED_SHUFFLE}:",
                run_dir,
            )
            model = make_model(
                source_dir,
                UNIVERSAL_DIMS,
                hparams["batch_size"],
                initial_lr=TUNED_INITIAL_LR,
                use_lr_finder=False,
                shuffle=TUNED_SHUFFLE,
            )
            model.load("best")
            model.cfg.directory = str(run_dir)
            model.fit(feff_splits[element])

## 8. Fine-tune UniversalXAS on VASP targets

In [ ]:
for element, do_train in VASP_TUNED_FLAGS.items():
    if not do_train or element not in vasp_splits:
        continue
    hparams = VASP_HPARAMS[element]
    source_dir = best_universal_source_for(f"{element} VASP tuning")
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        for dropout in TUNED_DROPOUT_SWEEP:
            seed_everything(train_seed, workers=True)
            XASBlock.DROPOUT = dropout
            run_dir = new_run_dir(run_root("tuned", element, "VASP"), train_seed, dropout)
            print(
                f"Fine-tuning {element} VASP Tuned-UniversalXAS "
                f"dropout={dropout} lr={TUNED_INITIAL_LR} shuffle={TUNED_SHUFFLE}:",
                run_dir,
            )
            model = make_model(
                source_dir,
                UNIVERSAL_DIMS,
                hparams["batch_size"],
                initial_lr=TUNED_INITIAL_LR,
                use_lr_finder=False,
                shuffle=TUNED_SHUFFLE,
            )
            model.load("best")
            model.cfg.directory = str(run_dir)
            model.fit(vasp_splits[element])

## 9. Best-eta table across saved checkpoints

In [ ]:
paper_eta = {
    ("Ti", "FEFF", "ExpertXAS"): 6.35,
    ("Ti", "FEFF", "UniversalXAS"): 4.19,
    ("Ti", "FEFF", "Tuned-UniversalXAS"): 7.63,
    ("V", "FEFF", "ExpertXAS"): 7.30,
    ("V", "FEFF", "UniversalXAS"): 5.19,
    ("V", "FEFF", "Tuned-UniversalXAS"): 9.22,
    ("Cr", "FEFF", "ExpertXAS"): 8.54,
    ("Cr", "FEFF", "UniversalXAS"): 7.13,
    ("Cr", "FEFF", "Tuned-UniversalXAS"): 10.44,
    ("Mn", "FEFF", "ExpertXAS"): 17.66,
    ("Mn", "FEFF", "UniversalXAS"): 13.15,
    ("Mn", "FEFF", "Tuned-UniversalXAS"): 29.81,
    ("Fe", "FEFF", "ExpertXAS"): 7.51,
    ("Fe", "FEFF", "UniversalXAS"): 6.04,
    ("Fe", "FEFF", "Tuned-UniversalXAS"): 8.98,
    ("Co", "FEFF", "ExpertXAS"): 14.47,
    ("Co", "FEFF", "UniversalXAS"): 9.58,
    ("Co", "FEFF", "Tuned-UniversalXAS"): 19.83,
    ("Ni", "FEFF", "ExpertXAS"): 8.45,
    ("Ni", "FEFF", "UniversalXAS"): 6.43,
    ("Ni", "FEFF", "Tuned-UniversalXAS"): 11.21,
    ("Cu", "FEFF", "ExpertXAS"): 5.19,
    ("Cu", "FEFF", "UniversalXAS"): 2.75,
    ("Cu", "FEFF", "Tuned-UniversalXAS"): 4.81,
    ("Ti", "VASP", "ExpertXAS"): 4.75,
    ("Ti", "VASP", "Tuned-UniversalXAS"): 5.27,
    ("Cu", "VASP", "ExpertXAS"): 8.46,
    ("Cu", "VASP", "Tuned-UniversalXAS"): 9.21,
}

model_specs = []
for element in FEFF_ELEMENTS:
    model_specs += [
        (element, "FEFF", "ExpertXAS", run_root("expert", element, "FEFF"), FEFF_HPARAMS[element]["widths"], FEFF_HPARAMS[element]["batch_size"], feff_splits[element]),
        (element, "FEFF", "UniversalXAS", run_root("universal"), UNIVERSAL_DIMS, 32, feff_splits[element]),
        (element, "FEFF", "Tuned-UniversalXAS", run_root("tuned", element, "FEFF"), UNIVERSAL_DIMS, FEFF_HPARAMS[element]["batch_size"], feff_splits[element]),
    ]
for element, split in vasp_splits.items():
    model_specs += [
        (element, "VASP", "ExpertXAS", run_root("expert", element, "VASP"), VASP_HPARAMS[element]["widths"], VASP_HPARAMS[element]["batch_size"], split),
        (element, "VASP", "Tuned-UniversalXAS", run_root("tuned", element, "VASP"), UNIVERSAL_DIMS, VASP_HPARAMS[element]["batch_size"], split),
    ]

rows = []
for element, spectrum_type, model_name, ckpt_root, hidden_dims, batch_size, split in model_specs:
    ckpts = sorted(ckpt_root.glob("paper_*/best*.ckpt"))
    if not ckpts:
        print(f"Skipping {element} {spectrum_type} / {model_name}: no checkpoints in {ckpt_root}")
        continue

    target = split.test.y
    baseline = np.repeat(split.train.y.mean(axis=0, keepdims=True), len(target), axis=0)
    baseline_median = float(np.median(np.mean((target - baseline) ** 2, axis=1)))

    ckpt = min(ckpts, key=checkpoint_val_loss)
    model = make_model(ckpt.parent, hidden_dims, batch_size)
    model.load("best")
    pred = predict_direct(model, split.test.X)
    metrics = ModelMetrics(predictions=pred, targets=target)
    median_mse = float(metrics.median_of_mse_per_spectra)
    eta = baseline_median / median_mse
    seed_match = re.search(r"seed(\d+)", ckpt.parent.name)
    row = {
        "notebook_seed": SEED,
        "checkpoint_seed": int(seed_match.group(1)) if seed_match else np.nan,
        "element": element,
        "type": spectrum_type,
        "dataset": f"{element} {spectrum_type}",
        "model": model_name,
        "mse": float(metrics.mse),
        "median_mse": median_mse,
        "baseline_median_mse": baseline_median,
        "eta": eta,
        "paper_eta": paper_eta.get((element, spectrum_type, model_name), np.nan),
        "val_loss": checkpoint_val_loss(ckpt),
    }
    rows.append(row)
    print(f"{element} {spectrum_type} / {model_name}: selected val_loss={row['val_loss']:.8g}, test eta={row['eta']:.6f}")

results = pd.DataFrame(rows)
display(results)
results.to_csv(RESULTS_DIR / "paper_style_all_elements_results.csv", index=False)
print("Saved:", RESULTS_DIR / "paper_style_all_elements_results.csv")